# BEXIMCO Next-Day Close Prediction

Updated pipeline:
- richer technical features
- chronological train / validation / test split
- **Ridge, Random Forest, XGBoost, LightGBM, CatBoost, HistGBM**
- **BiLSTM** and a **Transformer encoder** with *train-only* feature **and target** scaling
- stacked ensemble of the best tabular models

Neural nets previously underperformed mainly because the close price target was unscaled.

In [ ]:
# %pip install -q pandas numpy scikit-learn xgboost lightgbm catboost statsmodels tensorflow matplotlib

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import breaks_cusumolsresid
from statsmodels.tsa.stattools import adfuller

ROOT = Path.cwd()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "src" / "features.py").exists():
        sys.path.insert(0, str(candidate))
        break

from src.features import FEATURE_COLUMNS, add_technical_features
from src.metrics import regression_metrics
from src.models import build_bilstm, build_tabular_models, build_transformer, create_sequences
from src.split import time_split

In [ ]:
def load_cleaned():
    for path in [
        Path("data/Cleaned_DSE_Data.csv"),
        Path("../data/Cleaned_DSE_Data.csv"),
        Path("Cleaned.csv"),
        Path("Cleaned_DSE_Data.csv"),
        Path("../Cleaned.csv"),
    ]:
        if path.exists():
            print("Loaded", path)
            return pd.read_csv(path, parse_dates=["Date"])
    raise FileNotFoundError("Run the cleaning notebook first, or place Cleaned.csv next to this notebook.")


df = load_cleaned()
stock = "BEXIMCO" if "BEXIMCO" in set(df["Trading_Code"]) else df["Trading_Code"].value_counts().idxmax()
print("Using ticker:", stock)

bex = df[df["Trading_Code"] == stock].sort_values("Date").reset_index(drop=True)
bex = add_technical_features(bex)
bex = bex.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print("Rows after features:", len(bex))
bex.tail()

In [ ]:
feature_cols = [c for c in FEATURE_COLUMNS if c in bex.columns]
train_df, val_df, test_df = time_split(bex, train_ratio=0.70, val_ratio=0.15)
print(len(train_df), "train |", len(val_df), "val |", len(test_df), "test")

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(train_df[feature_cols])
X_val = x_scaler.transform(val_df[feature_cols])
X_test = x_scaler.transform(test_df[feature_cols])

y_train = train_df["Target_Close"].to_numpy()
y_val = val_df["Target_Close"].to_numpy()
y_test = test_df["Target_Close"].to_numpy()
close_test = test_df["Close"].to_numpy()

X_fit = np.vstack([X_train, X_val])
y_fit = np.concatenate([y_train, y_val])

In [ ]:
models = build_tabular_models()
rows = []
preds = {}

for name, model in models.items():
    fit_kwargs = {}
    if name == "XGBoost":
        fit_kwargs = {"eval_set": [(X_val, y_val)], "verbose": False}
    elif name == "LightGBM":
        fit_kwargs = {"eval_set": [(X_val, y_val)]}
        try:
            from lightgbm import early_stopping
            fit_kwargs["callbacks"] = [early_stopping(80, verbose=False)]
        except Exception:
            pass
    elif name == "CatBoost":
        fit_kwargs = {"eval_set": (X_val, y_val)}

    try:
        model.fit(X_train, y_train, **fit_kwargs)
    except TypeError:
        model.fit(X_fit, y_fit)

    pred = model.predict(X_test)
    preds[name] = np.asarray(pred).ravel()
    metrics = regression_metrics(y_test, preds[name], close_today=close_test)
    metrics["Model"] = name
    rows.append(metrics)
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items() if k != "Model"})

tabular = pd.DataFrame(rows).set_index("Model")
tabular.sort_values("RMSE")

In [ ]:
# Weighted ensemble of the three lowest-RMSE tabular models
top3 = tabular.nsmallest(3, "RMSE").index.tolist()
weights = 1.0 / tabular.loc[top3, "RMSE"]
weights = weights / weights.sum()
ensemble_pred = sum(weights[m] * preds[m] for m in top3)
preds["Ensemble"] = ensemble_pred
ens_metrics = regression_metrics(y_test, ensemble_pred, close_today=close_test)
ens_metrics["Model"] = "Ensemble"
print("Top-3:", top3)
print("Ensemble", {k: round(v, 4) for k, v in ens_metrics.items() if k != "Model"})

## Sequence models

Targets are standardized on the **training window only**, then inverted after prediction. That is the main LSTM upgrade versus the original notebook.

In [ ]:
LOOKBACK = 30
y_scaler.fit(y_train.reshape(-1, 1))

X_all = x_scaler.transform(bex[feature_cols])
y_all_scaled = y_scaler.transform(bex["Target_Close"].to_numpy().reshape(-1, 1)).ravel()

X_seq, y_seq = create_sequences(X_all, y_all_scaled, LOOKBACK)
idx = np.arange(LOOKBACK, len(bex))
train_mask = idx < len(train_df)
val_mask = (idx >= len(train_df)) & (idx < len(train_df) + len(val_df))
test_mask = idx >= len(train_df) + len(val_df)

X_tr_s, y_tr_s = X_seq[train_mask], y_seq[train_mask]
X_va_s, y_va_s = X_seq[val_mask], y_seq[val_mask]
X_te_s, y_te_s = X_seq[test_mask], y_seq[test_mask]
y_te_price = bex["Target_Close"].to_numpy()[idx[test_mask]]
close_te_seq = bex["Close"].to_numpy()[idx[test_mask]]
print(X_tr_s.shape, X_va_s.shape, X_te_s.shape)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]

bilstm = build_bilstm(LOOKBACK, X_tr_s.shape[-1])
bilstm.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_va_s, y_va_s),
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

lstm_pred = y_scaler.inverse_transform(bilstm.predict(X_te_s, verbose=0)).ravel()
lstm_metrics = regression_metrics(y_te_price, lstm_pred, close_today=close_te_seq)
lstm_metrics["Model"] = "BiLSTM"
print(lstm_metrics)

In [ ]:
transformer = build_transformer(LOOKBACK, X_tr_s.shape[-1])
transformer.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_va_s, y_va_s),
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

tf_pred = y_scaler.inverse_transform(transformer.predict(X_te_s, verbose=0)).ravel()
tf_metrics = regression_metrics(y_te_price, tf_pred, close_today=close_te_seq)
tf_metrics["Model"] = "Transformer"
print(tf_metrics)

In [ ]:
results = pd.concat(
    [
        tabular.reset_index(),
        pd.DataFrame([ens_metrics, lstm_metrics, tf_metrics]),
    ],
    ignore_index=True,
)[["Model", "R2", "RMSE", "MAE", "MAPE", "Direction_Acc"]]
results = results.sort_values("RMSE").reset_index(drop=True)
display(results.round(4))

best_name = results.iloc[0]["Model"]
print("Best test RMSE:", best_name)

In [ ]:
plot_pred = preds.get(best_name)
plot_dates = test_df["Date"]
plot_actual = y_test
if best_name == "BiLSTM":
    plot_pred, plot_dates, plot_actual = lstm_pred, bex["Date"].to_numpy()[idx[test_mask]], y_te_price
elif best_name == "Transformer":
    plot_pred, plot_dates, plot_actual = tf_pred, bex["Date"].to_numpy()[idx[test_mask]], y_te_price
elif best_name == "Ensemble":
    plot_pred = ensemble_pred

plt.figure(figsize=(14, 5))
plt.plot(plot_dates, plot_actual, label="Actual next close", linewidth=1.2)
plt.plot(plot_dates, plot_pred, label=f"{best_name} predicted", linewidth=1.2)
plt.title(f"{stock}: actual vs predicted next-day close")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Statistical diagnostics on the raw close series

In [ ]:
close = bex["Close"]
adf = adfuller(close)
print("ADF statistic:", adf[0], "p-value:", adf[1])

y = close.diff().dropna()
x = sm.add_constant(close.shift(1).dropna())
pp_stat = sm.OLS(y.iloc[1:], x.iloc[1:]).fit().tvalues.iloc[1]
print("PP statistic (custom):", float(pp_stat))

X_chow = sm.add_constant(bex[["Open", "High", "Low", "Volume"]])
chow_model = sm.OLS(bex["Close"], X_chow).fit()
print("CUSUM:", breaks_cusumolsresid(chow_model.resid))

plt.figure(figsize=(12, 4))
plt.plot(bex["Date"], bex["Close"])
plt.title(f"{stock} closing price")
plt.show()